# Combining the methods for cuts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 25
plt.rcParams['axes.labelsize'] = 25
plt.rcParams['xtick.labelsize'] = 25
plt.rcParams['ytick.labelsize'] = 25
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'

from glob import glob

In [ ]:
def calculate_PAO_spectrum(e_eV):
    """
    Calculate the CR flux measured by PAO in 10 ** 17 eV < E < 10 ** 20 eV
    with the SD-750 and SD-1500.
    The formula of the spectrum is Equation (13) of
    Abreu et al., Eur. Phys. J. C 81, 966 (2021)
    (https://doi.org/10.1140/epjc/s10052-021-09700-w)
    """


    j_0 = 1.309e-18 # km^-2 yr^-1 sr^-1 eV^-1
    e_0 = 10 ** 18.5 # eV
    omega_01, omega_12, omega_23, omega_34 = 0.43, 0.05, 0.05, 0.05
    g_0, g_1, g_2, g_3, g_4 = 2.64, 3.298, 2.52, 3.08, 5.2
    e_01, e_12, e_23, e_34 = 1.24e17, 4.9e18, 1.4e19, 4.7e19 # eV

    j = j_0 * (e_eV/e_0) ** (-g_0) \
            * (1 + (e_eV/e_01) ** (1/omega_01)) ** ((g_0-g_1) * omega_01) \
            * (1 + (e_eV/e_12) ** (1/omega_12)) ** ((g_1-g_2) * omega_12) \
            * (1 + (e_eV/e_23) ** (1/omega_23)) ** ((g_2-g_3) * omega_23) \
            * (1 + (e_eV/e_34) ** (1/omega_34)) ** ((g_3-g_4) * omega_34) \
            / (1 + (e_0/e_01)  ** (1/omega_01)) ** ((g_0-g_1) * omega_01) \
            / (1 + (e_0/e_12)  ** (1/omega_12)) ** ((g_1-g_2) * omega_12) \
            / (1 + (e_0/e_23)  ** (1/omega_23)) ** ((g_2-g_3) * omega_23) \
            / (1 + (e_0/e_34)  ** (1/omega_34)) ** ((g_3-g_4) * omega_34)

    return j

## Extract data

In [ ]:
parasite_chi2_PWF_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/chi2_PWF_parasites.npy",
                    allow_pickle=True
                    ).item()
parasite_chi2_SWF_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/chi2_SWF_parasites.npy",
                    allow_pickle=True
                    ).item()
parasite_chi2_ADF_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/chi2_ADF_parasites.npy",
                    allow_pickle=True
                    ).item()
parasite_polarization_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/polar_parasites.npy",
                    allow_pickle=True
                    ).item()
parasite_theta_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/theta_parasites.npy",
                    allow_pickle=True
                    ).item()
parasite_phi_RUN = np.load(f"/sps/grand/jlavoisier/code/quantification/check_data/data/phi_parasites.npy",
                    allow_pickle=True
                    ).item()

RUN = np.array(list(parasite_chi2_PWF_RUN.keys()))

parasite_chi2_PWF = np.array([])
parasite_chi2_SWF = np.array([])
parasite_chi2_ADF = np.array([])
parasite_polarization = np.array([])
parasite_theta = np.array([])
parasite_phi = np.array([])

for run in RUN:
    parasite_chi2_PWF = np.concatenate((parasite_chi2_PWF, parasite_chi2_PWF_RUN[run]))
    parasite_chi2_SWF = np.concatenate((parasite_chi2_SWF, parasite_chi2_SWF_RUN[run]))
    parasite_chi2_ADF = np.concatenate((parasite_chi2_ADF, parasite_chi2_ADF_RUN[run]))
    parasite_polarization = np.concatenate((parasite_polarization, parasite_polarization_RUN[run]))
    parasite_theta = np.concatenate((parasite_theta, parasite_theta_RUN[run]))
    parasite_phi = np.concatenate((parasite_phi, parasite_phi_RUN[run]))

In [ ]:
pass_cluster = np.load(f"/sps/grand/jlavoisier/code/quantification/cluster_neighbour_quant/cluster_results/cluster_pass_5s_10deg.npy",
                           allow_pickle=True
                        ).item()

mask_parasite_cluster = np.array([])
for run in RUN:
    mask_parasite_cluster = np.concatenate((mask_parasite_cluster, pass_cluster[run][:len(parasite_theta_RUN[run])]))

## Extract sims

In [ ]:
path = f"/sps/grand/jlavoisier/code/quantification/check_sims/out_files/simsGP300NJ_theta_n_ant_energy.npy"

file_sims = np.load(path)

theta_tot = file_sims[0]
phi_tot = file_sims[1]
n_ant_trig_tot = file_sims[2]
energy_tot = file_sims[3]

theta = theta_tot[n_ant_trig_tot >= 5]
phi = phi_tot[n_ant_trig_tot >= 5]
energy = energy_tot[n_ant_trig_tot >= 5]

In [ ]:
# chi2 PWF
sim_path_PWF = sorted(glob("/sps/grand/jlavoisier/code/quantification/SWF_quant/out_files/batch/ZHAiReS/AN*PWF*"))
sims_chi2_PWF = np.array([])

for i in range(len(sim_path_PWF)):
    file_sims = np.load(sim_path_PWF[i])
    print("Number of simulated events in file ", sim_path_PWF[i], " : ", len(file_sims[0]))
    sims_chi2_PWF = np.append(sims_chi2_PWF, file_sims[1])

# chi2 SWF
sim_path_SWF = sorted(glob("/sps/grand/jlavoisier/code/quantification/SWF_quant/out_files/batch/ZHAiReS/AN_*SWF*"))
sims_chi2_SWF = np.array([])

for i in range(len(sim_path_SWF)):
    file_sims = np.load(sim_path_SWF[i])
    print("Number of simulated events in file ", sim_path_SWF[i], " : ", len(file_sims[0]))
    sims_chi2_SWF = np.append(sims_chi2_SWF, file_sims[1])

# chi2 ADF
sim_path_ADF = sorted(glob("/sps/grand/jlavoisier/code/quantification/ADF_quant/out_files/batch/AN_*"))
sims_chi2_ADF = np.array([])

for i in range(len(sim_path_ADF)):
    file_sims = np.load(sim_path_ADF[i])
    print("Number of simulated events in file ", sim_path_ADF[i], " : ", len(file_sims[0]))
    sims_chi2_ADF = np.append(sims_chi2_ADF, file_sims[1])

# polarization
sims_polarization = np.load("/sps/grand/jlavoisier/code/quantification/polarization_cut/out_files/ZHAiReSGP300AN_polar.npy")[0]
sims_info = np.load("/sps/grand/jlavoisier/code/quantification/polarization_cut/out_files/ZHAiReSGP300AN_event_info.npy")

sims_n_ant_trig = sims_info[0]
sims_polar_per_event = np.zeros(len(sims_n_ant_trig))

for i in range(len(sims_n_ant_trig)):
    polar_event_i = sims_polarization[int(np.sum(sims_n_ant_trig[:i])):int(np.sum(sims_n_ant_trig[:i+1]))]
    sims_polar_per_event[i] = np.median(polar_event_i[polar_event_i < 2])


In [ ]:
# weights

# weights_energy_events = np.zeros(len(energy))
# for i in range(len(energy)):
#     weights_energy_events[i] = calculate_PAO_spectrum(energy[i] * 1e9) # The weight for the energy bin

weights_energy_events = calculate_PAO_spectrum(energy * 1e9)
weights_zenith_events = np.sin(theta * np.pi / 180) * np.cos(theta * np.pi / 180) # The weight for the zenith bin
weights_events = weights_energy_events * weights_zenith_events

weights_distrib = 1 / np.log10(1/np.cos(theta * np.pi / 180)) # The weight for the zenith bin in the distribution
weights_events_2 = weights_events * weights_distrib

## Cut parameters

In [ ]:
chi2_PWF_threshold = 1.83e3
chi2_SWF_threshold = 7.6e2
chi2_ADF_threshold = 55
polarization_threshold = 0.25

## Cut efficiency

In [ ]:
mask_parasite_chi2_PWF = parasite_chi2_PWF < chi2_PWF_threshold
mask_parasite_chi2_SWF = parasite_chi2_SWF < chi2_SWF_threshold
mask_parasite_chi2_ADF = parasite_chi2_ADF < chi2_ADF_threshold
mask_parasite_polarization = parasite_polarization < polarization_threshold

mask_parasite_combined = mask_parasite_chi2_PWF & mask_parasite_chi2_SWF & mask_parasite_chi2_ADF & mask_parasite_polarization & mask_parasite_cluster

print(f"Number of surviving parasites: {np.sum(mask_parasite_combined)}")
print(f"Cut efficiency: {np.sum(mask_parasite_combined)/len(mask_parasite_combined)}")

In [ ]:
surv_parasite_theta = parasite_theta[mask_parasite_combined]
surv_parasite_phi = parasite_phi[mask_parasite_combined]

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='polar')
ax.set_theta_zero_location("N")

hist = ax.hist2d(surv_parasite_phi-np.pi/2,
                surv_parasite_theta*180/np.pi,
                bins=[180, 45],
                cmap='Oranges',
                # alpha=0.7,
                label='Surviving parasites',
                norm=mpl.colors.LogNorm(),
                rasterized=True
                )


plt.colorbar(hist[3], 
             ax=ax, 
             label='Surviving parasite Density',
            #  pad=0.08,
             shrink=0.8
             )
ax.set_ylim(0, 90)

ax.tick_params(axis="x",
               pad=9,
               )

# plt.legend(fontsize=25,
#            loc='center left', 
#            bbox_to_anchor=(-0.15, -0.15, 1, 0.1),
#            ncol=2,
#            columnspacing=1,
#            handletextpad=0.4,
#            handlelength=1,
#            frameon=False
#            )

plt.savefig(f"/sps/grand/jlavoisier/code/quantification/0_plots_article/combined_polar_distrib_{len(surv_parasite_theta)}-cutparasites.pdf",
            format='pdf', 
            dpi=300, 
            bbox_inches = "tight"
            )


## Loss rate

Couldn't associate the loss rate with the cluster cut, as it was estiated on another data sample. Thus, we can only estimate the cut efficiency, which is the fraction of events that pass the cuts. The loss rate is the fraction of events that are lost due to the cuts.

In [ ]:
mask_CR_chi2_PWF = sims_chi2_PWF < chi2_PWF_threshold
mask_CR_chi2_SWF = sims_chi2_SWF < chi2_SWF_threshold
mask_CR_chi2_ADF = sims_chi2_ADF < chi2_ADF_threshold
mask_CR_polarization = sims_polar_per_event < polarization_threshold

mask_CR_combined = mask_CR_chi2_PWF & mask_CR_chi2_SWF & mask_CR_chi2_ADF & mask_CR_polarization

print(f"Number of surviving CR: {np.sum(mask_CR_combined)}")
print(f"CR loss rate: {np.sum(weights_events_2[mask_CR_combined])/np.sum(weights_events_2)}")